#### Forecast API

##### Llamada a la API Open Meteo Forecast

In [1]:
import openmeteo_requests
import requests_cache
import pandas as pd
import numpy as np
from retry_requests import retry
from pathlib import Path

def main():
    # 1) Forecast meteo
    cache = requests_cache.CachedSession('.cache', expire_after=3600)
    sess  = retry(cache, retries=5, backoff_factor=0.2)
    client = openmeteo_requests.Client(session=sess)

    vars_hr = [  # tu lista completa
        "temperature_2m", "dew_point_2m", "relative_humidity_2m", "apparent_temperature",
        "surface_pressure", "cloud_cover", "cloud_cover_low", "cloud_cover_mid", "visibility",
        "evapotranspiration", "et0_fao_evapotranspiration", "vapour_pressure_deficit", "wind_speed_10m",
        "wind_speed_80m", "wind_speed_120m", "wind_speed_180m", "wind_direction_10m", "wind_direction_80m",
        "wind_direction_120m", "wind_direction_180m", "wind_gusts_10m", "temperature_80m", "temperature_120m",
        "temperature_180m", "soil_temperature_0cm", "soil_temperature_6cm", "soil_temperature_18cm",
        "soil_temperature_54cm", "soil_moisture_0_to_1cm", "soil_moisture_1_to_3cm", "soil_moisture_3_to_9cm",
        "soil_moisture_9_to_27cm", "soil_moisture_27_to_81cm", "uv_index", "uv_index_clear_sky", "is_day",
        "sunshine_duration", "wet_bulb_temperature_2m", "cape", "lifted_index", "convective_inhibition",
        "freezing_level_height", "shortwave_radiation", "diffuse_radiation",
        "global_tilted_irradiance", "shortwave_radiation_instant", "diffuse_radiation_instant",
        "global_tilted_irradiance_instant", "direct_radiation", "direct_normal_irradiance",
        "terrestrial_radiation", "direct_radiation_instant", "direct_normal_irradiance_instant",
        "terrestrial_radiation_instant", "pressure_msl"
    ]

    params = {
        "latitude": 18.2158,
        "longitude": -71.0998,
        "hourly": vars_hr,
        "timezone": "UTC",
        "past_days": 1,
        "forecast_days": 7,
        "models": "best_match"
    }

    url = "https://api.open-meteo.com/v1/forecast"
    resps = client.weather_api(url, params=params)
    if not resps:
        raise RuntimeError("No response from forecast API")
    hr = resps[0].Hourly()
    t0 = pd.to_datetime(hr.Time(),      unit="s", utc=True)
    t1 = pd.to_datetime(hr.TimeEnd(),   unit="s", utc=True)
    freq = pd.Timedelta(seconds=hr.Interval())
    idx  = pd.date_range(t0, t1, freq=freq, inclusive="left", tz="UTC")

    df_m = pd.DataFrame(
        {v: hr.Variables(i).ValuesAsNumpy() for i,v in enumerate(vars_hr)},
        index=idx
    )

    # 2) Load histórico de generación
    hist_file = Path("../data/interim/post_despacho_transformed_data/post_despacho_transformed.parquet")
    df_h = pd.read_parquet(hist_file)
    # detecta la columna de generación
    gen_cols = [c for c in df_h.columns if "gen" in c.lower()]
    if not gen_cols:
        raise KeyError(f"No generation column found in {hist_file}")
    gen_col = gen_cols[0]
    df_h = df_h[[gen_col]].rename(columns={gen_col:"generation"})
    # asegurar índice datetime UTC
    if "timestamp" in df_h.columns:
        df_h["timestamp"] = pd.to_datetime(df_h["timestamp"], utc=True)
        df_h = df_h.set_index("timestamp")
    else:
        df_h.index = pd.to_datetime(df_h.index, utc=True)

    # 3) Extraer 24h previas al forecast
    start_fcst = idx.min()
    df_hist24  = df_h.loc[start_fcst - pd.Timedelta(days=1): start_fcst - pd.Timedelta(hours=1)]

    # 4) Construir columna generation: histórico + ceros
    gen = np.zeros(len(df_m), dtype=float)
    # ubica las horas históricas en el índice
    mask = df_m.index.isin(df_hist24.index)
    gen[mask] = df_hist24.reindex(df_m.index[mask]).generation.values
    df_m["generation"] = gen

    # 5) Save combined raw
    out = Path("../data/raw/forecast_meteo_data/parque_solar_girasol_forecast_api_request.csv")
    out.parent.mkdir(parents=True, exist_ok=True)
    df_m.reset_index().rename(columns={"index":"date"}).to_csv(out, index=False)
    print("✅ Saved meteo+generation raw to:", out)

if __name__ == "__main__":
    main()


✅ Saved meteo+generation raw to: ..\data\raw\forecast_meteo_data\parque_solar_girasol_forecast_api_request.csv


In [24]:
#### Forecast Feature Engineering – bloque único (ajustado)

import os
import pandas as pd
import joblib

# 1) Cargo el transformer fiteado y sus feature names
fe_path    = "../models/solar_feature_engineer.joblib"
sfe        = joblib.load(fe_path)
orig_feats = list(sfe.get_feature_names_out())

# Si por alguna razón tu lista contiene "date", lo eliminamos:
orig_feats = [f for f in orig_feats if f != "date"]

print(f"[INFO] Transformer cargado, {len(orig_feats)} features esperados")

# 2) Cargo el CSV raw y ajusto timezone
fcst_csv = "../data/raw/forecast_meteo_data/parque_solar_girasol_forecast_api_request.csv"
df_raw   = pd.read_csv(fcst_csv, parse_dates=["date"])

# localizo o convierto a UTC según venga
if df_raw["date"].dt.tz is None:
    df_raw["date"] = df_raw["date"].dt.tz_localize("UTC")
else:
    df_raw["date"] = df_raw["date"].dt.tz_convert("UTC")

df_raw = df_raw.set_index("date")

# 3) Selecciono SOLO las columnas crudas que entrenaste ("generation" + variables meteo)
#    Tomo la lista de tu histórico limpio
df_clean = pd.read_parquet(
    "../data/interim/meteo_data_with_generation_clean/parque_solar_girasol_clean.parquet"
)
raw_vars = list(df_clean.select_dtypes("number").columns.drop("generation"))
df_raw   = df_raw[ raw_vars + ["generation"] ]
print(f"[INFO] Raw forecast filtrado: {df_raw.shape}")

# 4) Transformo TODO el bloque con el mismo transformer (no fit)
df_all = sfe.transform(df_raw)
print(f"[INFO] Features generadas (hist+fcst): {df_all.shape}")

# 5) Re-alineo a las features EXACTAS del training
#    (ahora orig_feats no tiene "date", deberían coincidir)
df_all = df_all[ orig_feats ]
print(f"[INFO] Alineado a {df_all.shape[1]} features de training")

# 6) Extraigo solo el período de forecast (donde generation==0)
df_fcst = df_all[df_all["generation"] == 0]
print(f"[INFO] Solo forecast ready: {df_fcst.shape}")

# 7) Guardo parquet final
out_dir  = "../data/processed"
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "parque_solar_girasol_forecast_model_ready.parquet")
df_fcst.to_parquet(out_path)
print("✅ Guardado:", out_path)

[INFO] Transformer cargado, 403 features esperados
[INFO] Raw forecast filtrado: (216, 30)
[INFO] Features generadas (hist+fcst): (192, 417)
[INFO] Alineado a 403 features de training
[INFO] Solo forecast ready: (192, 403)
✅ Guardado: ../data/processed\parque_solar_girasol_forecast_model_ready.parquet


C:\Users\ferna\AppData\Local\Temp\ipykernel_3216\38042949.py:84: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_roll{w}h"] = df[col].rolling(window=w, min_periods=1).mean()
C:\Users\ferna\AppData\Local\Temp\ipykernel_3216\38042949.py:84: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_roll{w}h"] = df[col].rolling(window=w, min_periods=1).mean()
C:\Users\ferna\AppData\Local\Temp\ipykernel_3216\38042949.py:84: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.inser

In [32]:
import numpy as np
import pandas as pd
from joblib import load

# 1) Cargo las features ya procesadas (X ready)
X = pd.read_parquet(
    "../data/processed/parque_solar_girasol_forecast_model_ready.parquet"
).reset_index(drop=True)

# 2) Cargo el CSV raw de forecast
raw = pd.read_csv(
    "../data/raw/forecast_meteo_data/parque_solar_girasol_forecast_api_request.csv",
    low_memory=False
)

# 3) Detecto y renombro la columna de fecha a "datetime"
date_cols = [c for c in raw.columns if c.lower() == "date" or "time" in c.lower() or "fecha" in c.lower()]
if not date_cols:
    raise ValueError(f"No encontré columna de fecha. Columnas disponibles: {raw.columns.tolist()}")
raw[date_cols[0]] = pd.to_datetime(raw[date_cols[0]])
raw = raw.rename(columns={date_cols[0]: "datetime"})

# 4) Corto las primeras 24h (max_lag) para alinear con las X
max_lag = 24
raw2 = raw.iloc[max_lag:].reset_index(drop=True)
X = X.iloc[: len(raw2)].reset_index(drop=True)

# 5) Cargo el modelo
model = load("../models/solar_generation_model.joblib")

# 6) Aseguro que X sólo tenga las columnas que el modelo vio en fit
if hasattr(model, "feature_names_in_"):
    expected = list(model.feature_names_in_)
    missing = set(expected) - set(X.columns)
    extra = set(X.columns) - set(expected)
    if missing:
        raise ValueError(f"Faltan columnas en X: {missing}")
    X = X[expected]
else:
    # si no tiene feature_names_in_, quitamos manualmente 'generation' si está
    X = X.drop(columns=["generation"], errors="ignore")

# 7) Predicción cruda y cero en nocturno
pred = model.predict(X)
if "is_day" not in raw2.columns:
    raise KeyError(f"No encontré 'is_day' en raw. Columnas: {raw2.columns.tolist()}")
pred = np.where(raw2["is_day"] == 1, pred, 0)

# 8) Preparo el DataFrame de salida y aplico shift de 5h
df_out = pd.DataFrame({
    "datetime": raw2["datetime"],
    "predicted_generation": pred
})
df_out["predicted_generation"] = df_out["predicted_generation"].shift(-5)

# 9) Quito NaN y guardo en CSV
df_out = df_out.dropna(subset=["predicted_generation"]).reset_index(drop=True)
df_out.to_csv(
    "../data/processed/parque_solar_girasol_predicciones_alineadas.csv",
    index=False
)

print("✅ Guardado CSV alineado. Primeras filas:")
print(df_out.head())

✅ Guardado CSV alineado. Primeras filas:
                   datetime  predicted_generation
0 2025-05-11 00:00:00+00:00                   0.0
1 2025-05-11 01:00:00+00:00                   0.0
2 2025-05-11 02:00:00+00:00                   0.0
3 2025-05-11 03:00:00+00:00                   0.0
4 2025-05-11 04:00:00+00:00                   0.0


In [1]:
import openmeteo_requests
import requests_cache
from retry_requests import retry
import pandas as pd # Still useful for understanding time if you want to inspect

def get_forecast_model(latitude, longitude, past_days=1, forecast_days=2, timezone="auto"):
    """
    Makes a call to the Open-Meteo API and returns the model used for the forecast.
    """
    # Setup caching and retry mechanism
    cache = requests_cache.CachedSession('.cache', expire_after=3600)
    sess = retry(cache, retries=5, backoff_factor=0.2)
    client = openmeteo_requests.Client(session=sess)

    # Define the variables you want (can be minimal for just checking the model)
    # Even if you don't care about the data, you need to request at least one variable.
    vars_hr = ["temperature_2m"]

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": vars_hr,
        "timezone": timezone,
        "past_days": past_days,
        "forecast_days": forecast_days,
        # Using "best_match" explicitly, or omit for default behavior which is usually best_match
        "models": "best_match"
    }

    url = "https://api.open-meteo.com/v1/forecast"
    resps = client.weather_api(url, params=params)

    if not resps:
        raise RuntimeError("No response from forecast API")

    # The model information is available directly on the response object
    # For a single request, resps will be a list with one element.
    forecast_response = resps[0]
    model_used = forecast_response.Model()

    return model_used

if __name__ == "__main__":
    # Example usage with your provided coordinates
    latitude = 18.2158
    longitude = -71.0998
    
    # You can try with "auto" or "UTC" for timezone
    # The 'models' parameter in the API call determines which model Open-Meteo tries to use.
    # "best_match" is their default and recommended for most cases.
    # The 'model' attribute in the response tells you which one was actually used.
    
    try:
        model = get_forecast_model(latitude, longitude, timezone="UTC")
        print(f"The Open-Meteo API used the model: {model}")
        
        # If you want to see what happens with "auto" timezone
        model_auto_tz = get_forecast_model(latitude, longitude, timezone="auto")
        print(f"With timezone=auto, the Open-Meteo API used the model: {model_auto_tz}")

    except Exception as e:
        print(f"An error occurred: {e}")

The Open-Meteo API used the model: 1
With timezone=auto, the Open-Meteo API used the model: 1
